# Qwen3.5-4B → OpenVINO Split-IR + HuggingFace Push

Converts Qwen3.5-4B weights to a **split-IR** OpenVINO representation (one `.xml/.bin`
per layer) using direct `ov.convert_model()` — **no ONNX step**, no Dynamo issues.

| Step | Action |
|------|--------|
| 0 | Load Kaggle secrets |
| 1 | `git clone dsainvg/openvino-model-conv` |
| 2 | Install requirements (torch CPU + openvino + huggingface_hub) |
| 3 | `pytest qwen35/tests/` — toy smoke tests (no weights needed) |
| 4 | `qwen35/scripts/download_model.py` — pull weights from HF |
| 5 | `qwen35/scripts/convert_to_openvino.py` — split-IR conversion |
| 6 | `qwen35/scripts/push_to_hf.py` — upload IR to HuggingFace |

> **Kaggle Secrets needed** (Add-ons → Secrets):
> - `HF_TOKEN` — HuggingFace token with **write** scope
> - `HF_REPO_NAME` — target repo *(default: `qwen35-4b-openvino-split-ir`)*

---
**Architecture**: Each decoder layer is exported as a separate IR to stay within
Kaggle's RAM limits. At inference time the Python orchestrator chains them.

```
embed.xml        (input_ids) → (hidden_states)
layer_0.xml      hidden + linear-attn state → hidden + new state
layer_1.xml      ...
layer_3.xml      hidden + KV-cache + cos/sin → hidden + updated KV-cache
...  (32 layers total)
lm_head.xml      (hidden_states) → (logits)
```

## 0 · Secrets

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    def _get(key, fallback=None):
        try:
            return _s.get_secret(key)
        except Exception:
            return fallback
except ImportError:
    def _get(key, fallback=None):
        return os.environ.get(key, fallback)

HF_TOKEN     = _get("HF_TOKEN")
HF_REPO_NAME = _get("HF_REPO_NAME", "qwen35-4b-openvino-split-ir")

if not HF_TOKEN:
    raise EnvironmentError("HF_TOKEN secret is missing. Add it under Add-ons → Secrets.")

os.environ["HF_TOKEN"]               = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

print(f"HF_REPO_NAME : {HF_REPO_NAME}")
print("HF_TOKEN     : *** (set)")

## 1 · Clone the converter repo

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/dsainvg/openvino-model-conv.git"
REPO_DIR = Path("/kaggle/working/openvino-model-conv")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, str(REPO_DIR)], check=True)

QWEN35_DIR  = REPO_DIR / "qwen35"
SCRIPTS_DIR = QWEN35_DIR / "scripts"
MODEL_DIR   = Path("/kaggle/working/Qwen3.5-4B")
OUTPUT_DIR  = Path("/kaggle/working/ov_ir_qwen35_4b")

print(f"Repo    : {REPO_DIR}")
print(f"Scripts : {SCRIPTS_DIR}")

## 2 · Install requirements

No ONNX packages needed — conversion goes directly `nn.Module → ov.convert_model()`.

In [ ]:
def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

# CPU-only torch to avoid CUDA version conflicts with Kaggle's cuDNN stack
pip("torch", "--index-url", "https://download.pytorch.org/whl/cpu")
pip(
    "openvino",          # ov.convert_model / ov.save_model
    "huggingface_hub",   # hf_hub_download / push_to_hub
    "safetensors",       # safe_open for weight loading
    "sentencepiece",     # tokenizer
    "tiktoken",          # tokenizer (some checkpoints)
    "accelerate",        # optional: used by transformers for fast loading
    "pytest",            # smoke tests
)

print("Done.")

## 3 · Toy smoke tests (`qwen35/tests/`)

Runs two test suites against tiny random-weight models — **no downloads needed**.
- `test_toy_match.py` — unit tests for every modeling primitive
- `test_end_to_end.py` — runs the full split-IR conversion pipeline on a toy model
  (calls `main()` in-process; no subprocess segfault risk)

All 11 tests should pass in under 30 seconds.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--tb=short"],
    cwd=str(QWEN35_DIR),
    env={**os.environ},
)
if result.returncode != 0:
    raise RuntimeError("Toy smoke tests FAILED — fix modeling code before converting real weights.")
print("\n✓ All toy tests passed.")

## 4 · Download Qwen3.5-4B weights

Downloads BF16 safetensors from `Qwen/Qwen3.5-4B` on HuggingFace.
Requires `HF_TOKEN` with read access.

In [ ]:
result = subprocess.run(
    [
        sys.executable, str(SCRIPTS_DIR / "download_model.py"),
        "--model",  "Qwen/Qwen3.5-4B",
        "--output", str(MODEL_DIR),
        "--token",  HF_TOKEN,
    ],
    cwd=str(QWEN35_DIR),
    env={**os.environ},
)
if result.returncode != 0:
    raise RuntimeError("download_model.py failed.")
print("\n✓ Model downloaded to", MODEL_DIR)

## 5 · Convert to OpenVINO Split-IR

Loads BF16 weights **one layer at a time** (low-RAM), converts each via
`ov.convert_model(wrapper, example_input=...)` and saves:

```
ov_ir_qwen35_4b/
  embed.xml  +  embed.bin
  layer_0.xml  +  layer_0.bin
  ...  (32 layers)
  lm_head.xml  +  lm_head.bin
  tokenizer_config.json  config.json  ...
```

Expected runtime: **~60–90 min** on Kaggle CPU (most time is weight I/O + OV capture).

In [ ]:
result = subprocess.run(
    [
        sys.executable, str(SCRIPTS_DIR / "convert_to_openvino.py"),
        "--model-dir", str(MODEL_DIR),
        "--output",    str(OUTPUT_DIR),
        "--dtype",     "bf16",
        # NOTE: --compile-check is optional; skip on Kaggle to save time.
        # Uncomment to verify IR loads correctly on CPU:
        # "--compile-check",
    ],
    cwd=str(QWEN35_DIR),
    env={**os.environ},
)
if result.returncode != 0:
    raise RuntimeError("convert_to_openvino.py failed.")
print("\n✓ Split-IR saved to", OUTPUT_DIR)

# Quick sanity check: list output files
import os as _os
xmls = sorted(Path(OUTPUT_DIR).glob("*.xml"))
print(f"  {len(xmls)} IR files: {[x.name for x in xmls[:5]]} ...")

## 6 · Push to HuggingFace

Uploads all `.xml` + `.bin` files plus tokenizer/config to your HF repo.
Requires `HF_TOKEN` with **write** scope.

In [ ]:
result = subprocess.run(
    [
        sys.executable, str(SCRIPTS_DIR / "push_to_hf.py"),
        "--ir-dir",    str(OUTPUT_DIR),
        "--repo-name", HF_REPO_NAME,
        "--token",     HF_TOKEN,
    ],
    cwd=str(QWEN35_DIR),
    env={**os.environ},
)
if result.returncode != 0:
    raise RuntimeError("push_to_hf.py failed.")
print("\n✓ Uploaded to HuggingFace:", HF_REPO_NAME)